# BUSI · разведочный анализ в Google Colab
Откройте этот ноутбук в Colab. На Google Drive разместите рабочую копию проекта с текущим split_v2 и распакованный BUSI. Укажите их пути в первой ячейке и выполните все ячейки.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

from google.colab import drive
import subprocess
import sys

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/colab/breast-cancer-classification")
root = Path("/content/drive/MyDrive/colab/data/busi")
art = project / "artifacts"
if not (art / "splits/split_v2.csv").is_file():
    raise FileNotFoundError(f"Укажите путь к рабочей копии с split_v2: {project}")
os.chdir(project)
subprocess.run([sys.executable, "-m", "busi.prepare", "--root", str(root)], check=True)
out = art / "eda"
out.mkdir(exist_ok=True)
# Читаем результаты подготовки: свойства снимков и обнаруженные проблемы.
df = pd.read_csv(art / "manifest.csv")
audit = pd.read_csv(art / "audit.csv")
classes = ["normal", "benign", "malignant"]
counts = df["class"].value_counts().reindex(classes)

display(pd.DataFrame({"Снимки": counts, "Доля": counts / counts.sum()}))

def save(fig, name):
    """Сохранить график в artifacts/eda, показать его и освободить ресурсы."""
    fig.tight_layout()
    fig.savefig(out / name, dpi=125)
    plt.show()
    plt.close(fig)

In [ ]:
# Число снимков и доли классов показывают степень дисбаланса датасета.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
counts.plot.bar(ax=axes[0], title="Количество снимков", rot=0)

(counts / counts.sum() * 100).plot.bar(ax=axes[1], title="Доля снимков, %", rot=0)
save(fig, "classes.png")
# Размеры и число каналов помогают выбрать общую предобработку входов.

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
df[["width","height"]].plot.hist(ax=axes[0], alpha=.5, bins=25, title="Размеры, пиксели")
df.channels.value_counts().sort_index().plot.bar(ax=axes[1], title="Число каналов", rot=0)

save(fig, "dimensions.png")

In [ ]:
def gallery(rows, name):
    """Показать выбранные снимки в оттенках серого и сохранить галерею."""
    # squeeze=False сохраняет двумерный массив осей даже для одного снимка.
    fig, axes = plt.subplots(1, len(rows), figsize=(4 * len(rows), 3), squeeze=False)
    for ax, row in zip(axes.flat, rows.itertuples()):
        with Image.open(root / row.filepath) as im:
            ax.imshow(im.convert("L"), cmap="gray", vmin=0, vmax=255)
        ax.set_title(Path(row.filepath).name)
        ax.axis("off")
    save(fig, name)

# Фиксируем случайный выбор примера каждого класса для повторяемости обзора.
gallery(df.groupby("class").sample(1, random_state=42), "examples.png")
# Крайние размеры кадров полезны для ручной проверки качества и эффекта resize.
review = df.loc[[df.width.idxmin(), df.width.idxmax(), df.height.idxmin(), df.height.idxmax()]]
gallery(review.drop_duplicates("filepath"), "quality.png")
display(audit.issue.value_counts())
display(Markdown(f"Самый частый класс **{counts.idxmax()}** встречается в {counts.max()/counts.min():.2f} раза чаще самого редкого **{counts.idxmin()}**. Для сравнения моделей используем macro-F1 и показатели каждого класса."))